<a href="https://colab.research.google.com/github/sivaasudarson2002-sudo/ship-fleet-copilot/blob/main/Fleet_Copilot_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Fleet Copilot — AI agent on ship voyage data
Run cells one by one. Needs `ship_clustered.csv`, `ship_clusters.joblib` and the `GEMINI_API_KEY` Colab Secret.

## Cell 1 — Upload the 2 files from Day 2

In [1]:
from google.colab import files
uploaded = files.upload()   # select ship_clustered.csv AND ship_clusters.joblib together

Saving ship_clustered.csv to ship_clustered.csv
Saving ship_clusters.joblib to ship_clusters.joblib


## Cell 2 — Load data + cluster model, name the 4 profiles

In [7]:
import pandas as pd
import joblib

fleet = pd.read_csv("ship_clustered.csv")
fleet = fleet.rename(columns={"Cluster": "cluster"})
bundle = joblib.load("ship_clusters.joblib")
scaler, kmeans, features = bundle["scaler"], bundle["Kmeans"], bundle["features"]

profile = fleet.groupby("cluster")[["Efficiency_nm_per_kWh", "Operational_Cost_USD",
                                    "Average_Load_Percentage", "Profit_USD"]].mean()
best = profile["Efficiency_nm_per_kWh"].idxmax()
worst = profile["Efficiency_nm_per_kWh"].idxmin()
middle = profile.drop([best, worst])
half_empty = middle["Average_Load_Percentage"].idxmin()
full = middle["Average_Load_Percentage"].idxmax()

CLUSTER_INFO = {
    best: ("Efficient money-makers", "Study and copy their speed, routes and loading practices across the fleet."),
    full: ("Full but costly", "Investigate high costs: fuel price, port fees, delays and charter rates."),
    half_empty: ("Half-empty and costly", "Improve cargo planning to fill unused capacity or combine voyages."),
    worst: ("Cheap but fuel-hungry", "Schedule hull cleaning, propeller polishing and engine checks to cut energy use."),
}
fleet["profile"] = fleet["cluster"].map(lambda c: CLUSTER_INFO[c][0])
print(fleet.shape)
print(fleet["profile"].value_counts())

(2736, 25)
profile
Half-empty and costly     715
Cheap but fuel-hungry     690
Full but costly           680
Efficient money-makers    651
Name: count, dtype: int64


## Cell 3 — The agent's tools (normal Python functions)

In [8]:
import json

def to_plain(obj):
    """Convert numpy/pandas numbers to plain Python types so the result can be sent to Gemini."""
    return json.loads(json.dumps(obj, default=lambda o: o.item() if hasattr(o, "item") else str(o)))

CATEGORY_COLS = ["Ship_Type", "Route_Type", "Engine_Type", "Maintenance_Status", "Weather_Condition", "profile"]
METRIC_COLS = ["Efficiency_nm_per_kWh", "Profit_USD", "Operational_Cost_USD", "Revenue_per_Voyage_USD",
               "Energy_Used_kWh", "Speed_Over_Ground_knots", "Average_Load_Percentage", "Turnaround_Time_hours"]


def fleet_summary() -> dict:
    """Get an overview of the whole fleet dataset: number of voyages, date range, average profit,
    share of loss-making voyages, average efficiency and average energy used."""
    return to_plain({
        "voyages": len(fleet),
        "date_range": f"{fleet['Date'].min()} to {fleet['Date'].max()}",
        "avg_profit_usd": round(fleet["Profit_USD"].mean()),
        "loss_making_voyages": int((fleet["Profit_USD"] < 0).sum()),
        "loss_making_pct": round((fleet["Profit_USD"] < 0).mean() * 100, 1),
        "avg_efficiency_nm_per_kwh": round(fleet["Efficiency_nm_per_kWh"].mean(), 3),
        "avg_energy_used_kwh": round(fleet["Energy_Used_kWh"].mean()),
        "ship_types": fleet["Ship_Type"].value_counts().to_dict(),
    })


def compare_groups(group_by: str, metric: str) -> dict:
    """Compare the average of one metric across the categories of one column.

    Args:
        group_by: Column to group by. One of: Ship_Type, Route_Type, Engine_Type,
            Maintenance_Status, Weather_Condition, profile.
        metric: Number column to average. One of: Efficiency_nm_per_kWh, Profit_USD,
            Operational_Cost_USD, Revenue_per_Voyage_USD, Energy_Used_kWh,
            Speed_Over_Ground_knots, Average_Load_Percentage, Turnaround_Time_hours.
    """
    if group_by not in CATEGORY_COLS:
        return {"error": f"group_by must be one of {CATEGORY_COLS}"}
    if metric not in METRIC_COLS:
        return {"error": f"metric must be one of {METRIC_COLS}"}
    result = fleet.groupby(group_by)[metric].agg(["mean", "count"]).round(3)
    result = result.sort_values("mean", ascending=False)
    return to_plain({"group_by": group_by, "metric": metric,
                     "results": result.reset_index().to_dict(orient="records")})


def loss_making_voyages(group_by: str = "Ship_Type") -> dict:
    """Count loss-making voyages (profit below zero) and the loss rate for each category of a column.

    Args:
        group_by: One of: Ship_Type, Route_Type, Engine_Type, Maintenance_Status, Weather_Condition, profile.
    """
    if group_by not in CATEGORY_COLS:
        return {"error": f"group_by must be one of {CATEGORY_COLS}"}
    loss = fleet.assign(is_loss=fleet["Profit_USD"] < 0)
    table = loss.groupby(group_by)["is_loss"].agg(["sum", "mean"]).rename(
        columns={"sum": "loss_voyages", "mean": "loss_rate"})
    table["loss_rate"] = (table["loss_rate"] * 100).round(1)
    table = table.sort_values("loss_rate", ascending=False)
    return to_plain({"group_by": group_by, "total_loss_voyages": int(loss["is_loss"].sum()),
                     "results": table.reset_index().to_dict(orient="records")})


def get_profiles() -> list[dict]:
    """Get the 4 voyage operating profiles found by KMeans clustering, with their averages,
    number of voyages and the recommended action for each profile."""
    out = []
    for c, (name, action) in CLUSTER_INFO.items():
        group = fleet[fleet["cluster"] == c]
        out.append({
            "profile": name, "voyages": len(group),
            "avg_efficiency": round(group["Efficiency_nm_per_kWh"].mean(), 2),
            "avg_cost_usd": round(group["Operational_Cost_USD"].mean()),
            "avg_load_pct": round(group["Average_Load_Percentage"].mean(), 1),
            "avg_profit_usd": round(group["Profit_USD"].mean()),
            "recommended_action": action,
        })
    return to_plain(out)


def classify_voyage(speed_knots: float, distance_nm: float, cost_usd: float,
                    revenue_usd: float, efficiency_nm_per_kwh: float, load_pct: float) -> dict:
    """Assign a NEW voyage to one of the 4 operating profiles using the trained KMeans model,
    and return the recommended action.

    Args:
        speed_knots: Average speed over ground in knots (dataset range 10-25).
        distance_nm: Distance travelled in nautical miles (50-2000).
        cost_usd: Operational cost in USD (10,000-500,000).
        revenue_usd: Revenue for the voyage in USD (50,000-1,000,000).
        efficiency_nm_per_kwh: Efficiency in nautical miles per kWh (0.1-1.5).
        load_pct: Average load percentage (50-100).
    """
    row = pd.DataFrame([[speed_knots, distance_nm, cost_usd, revenue_usd,
                         efficiency_nm_per_kwh, load_pct]], columns=features)
    c = int(kmeans.predict(scaler.transform(row))[0])
    name, action = CLUSTER_INFO[c]
    return to_plain({"profile": name, "recommended_action": action,
            "estimated_profit_usd": round(revenue_usd - cost_usd),
            "estimated_energy_used_kwh": round(distance_nm / efficiency_nm_per_kwh)})


TOOLS = [fleet_summary, compare_groups, loss_making_voyages, get_profiles, classify_voyage]

## Cell 4 — Test each tool WITHOUT the AI

In [9]:
import json
for name, result in [("fleet_summary", fleet_summary()),
                     ("compare_groups", compare_groups("Weather_Condition", "Efficiency_nm_per_kWh")),
                     ("loss_making_voyages", loss_making_voyages("Ship_Type")),
                     ("get_profiles", get_profiles()[0]),
                     ("classify_voyage", classify_voyage(15, 1200, 150000, 600000, 1.3, 85))]:
    print(f"--- {name} ---")
    print(json.dumps(result, indent=2)[:600])
    print()

--- fleet_summary ---
{
  "voyages": 2736,
  "date_range": "2023-06-04 to 2024-06-30",
  "avg_profit_usd": 266219,
  "loss_making_voyages": 562,
  "loss_making_pct": 20.5,
  "avg_efficiency_nm_per_kwh": 0.799,
  "avg_energy_used_kwh": 1999,
  "ship_types": {
    "Bulk Carrier": 669,
    "Fish Carrier": 653,
    "Tanker": 643,
    "Container Ship": 635,
    "unknown": 136
  }
}

--- compare_groups ---
{
  "group_by": "Weather_Condition",
  "metric": "Efficiency_nm_per_kWh",
  "results": [
    {
      "Weather_Condition": "Calm",
      "mean": 0.825,
      "count": 893
    },
    {
      "Weather_Condition": "Rough",
      "mean": 0.795,
      "count": 816
    },
    {
      "Weather_Condition": "Moderate",
      "mean": 0.783,
      "count": 891
    },
    {
      "Weather_Condition": "unknown",
      "mean": 0.749,
      "count": 136
    }
  ]
}

--- loss_making_voyages ---
{
  "group_by": "Ship_Type",
  "total_loss_voyages": 562,
  "results": [
    {
      "Ship_Type": "Tanker",
     

## Cell 5 — Connect to Gemini and pick a working free model

In [10]:
from google import genai
from google.genai import types
from google.colab import userdata

API_KEY = userdata.get("GEMINI_API_KEY")
client = genai.Client(
    api_key=API_KEY,
    http_options=types.HttpOptions(timeout=60000,  # give up after 60 seconds, never hang
                                   retry_options=types.HttpRetryOptions(attempts=2)),
)

MODEL = None
for candidate in ["gemini-3.5-flash-lite", "gemini-3.6-flash", "gemini-3.8-flash", "gemini-3.1-flash-lite"]:
    try:
        reply = client.models.generate_content(model=candidate, contents="Reply with the single word: ready")
        print(f"OK   {candidate}: {reply.text.strip()}")
        MODEL = candidate
        break
    except Exception as e:
        print(f"FAIL {candidate}: {str(e)[:120]}")
print("Using model:", MODEL)

OK   gemini-3.5-flash-lite: ready
Using model: gemini-3.5-flash-lite


## Cell 6 — Create the agent (LLM + tools + rules)

In [11]:
SYSTEM_PROMPT = """You are Fleet Copilot, an AI assistant for a shipping company's operations team.
You answer questions about the company's voyage data (2,736 weekly voyage records).

Rules:
- ALWAYS use the tools to get numbers. Never invent or estimate numbers yourself.
- Use as few tool calls as possible; prefer one call that answers the whole question.
- Explain results in simple business language with short bullet points.
- When useful, give a practical recommendation (e.g. hull cleaning, cargo planning, cost review).
- If differences between groups are very small, say they are not meaningful.
- If a question cannot be answered with the tools, say so honestly.
"""

tool_log = []

def logged(fn):
    """Wrap a tool so every call is recorded (to show the agent's steps)."""
    import functools
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        result = fn(*args, **kwargs)
        tool_log.append((fn.__name__, kwargs))
        return result
    return wrapper

chat = client.chats.create(
    model=MODEL,
    config=types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        tools=[logged(t) for t in TOOLS],
        temperature=0.2,
        automatic_function_calling=types.AutomaticFunctionCallingConfig(maximum_remote_calls=6),
    ),
)
print("Fleet Copilot ready with", len(TOOLS), "tools")

Fleet Copilot ready with 5 tools


## Cell 7 — Helper to ask questions and show the agent's steps

In [12]:
from IPython.display import Markdown, display

def ask(question):
    tool_log.clear()
    try:
        response = chat.send_message(question)
    except Exception as e:
        print("Error:", str(e)[:300])
        print("Tip: '429' = free limit reached, wait 1 minute. 'timeout' = run the cell again.")
        return
    print("You:", question)
    for i, (name, args) in enumerate(tool_log, 1):
        print(f"  Step {i}: {name}({args})")
    display(Markdown("**Fleet Copilot:**\n\n" + (response.text or "(no text returned)")))

## Cell 8 — Ask questions (run ONE at a time, wait ~15 s between)

In [13]:
ask("Give me a quick overview of the fleet.")

You: Give me a quick overview of the fleet.
  Step 1: fleet_summary({})


**Fleet Copilot:**

Here is a quick overview of the fleet based on all 2,736 weekly voyage records from June 2023 to June 2024:

* **Total Voyages:** 2,736 tracked across Bulk Carriers, Container Ships, Fish Carriers, and Tankers.
* **Average Profit:** $266,219 per voyage.
* **Loss-Making Voyages:** 20.5% of voyages (562 voyages) operated at a loss.
* **Average Efficiency:** 0.80 nautical miles per kWh, with an average energy usage of 1,999 kWh per voyage.

**Operational Recommendation:**
With over one in five voyages losing money, consider investigating the loss-making trips—particularly by examining route types, maintenance statuses, and operating profiles to identify cost reduction opportunities.

In [14]:
ask("Which ship type loses money most often?")

You: Which ship type loses money most often?
  Step 1: loss_making_voyages({'group_by': 'Ship_Type'})


**Fleet Copilot:**

Here is the breakdown of loss-making voyages by ship type:

* **Tankers** have the highest loss rate at **22.6%** (145 loss-making voyages).
* **Container Ships** follow closely at **21.3%** (135 voyages).
* **Bulk Carriers** are at **20.3%** (136 voyages).
* **Fish Carriers** have the lowest loss rate among known types at **18.7%** (122 voyages).

**Recommendation:** 
Since Tankers and Container Ships show the highest susceptibility to losses, review their operational costs, fuel efficiency, and revenue structures to protect margins.

In [15]:
ask("Does weather affect efficiency?")

You: Does weather affect efficiency?
  Step 1: compare_groups({'group_by': 'Weather_Condition', 'metric': 'Efficiency_nm_per_kWh'})


**Fleet Copilot:**

Weather conditions have a relatively small impact on fuel/energy efficiency:

* **Calm weather** yields the highest efficiency at an average of **0.825 nm/kWh**.
* **Rough weather** averages **0.795 nm/kWh**.
* **Moderate weather** averages **0.783 nm/kWh**.

**Recommendation:** 
While rough and moderate weather slightly reduce efficiency, the differences are minor. Focus optimization efforts on other operational factors (such as speed and maintenance) where larger efficiency gains can be achieved.

In [17]:
ask("What is the loss percentage?")

You: What is the loss percentage?
  Step 1: fleet_summary({})


**Fleet Copilot:**

The overall loss percentage for the fleet is **20.5%**, which represents 562 out of 2,736 total voyages operating at a negative profit.

In [18]:
ask("A new voyage: speed 15 knots, 1200 nm, cost 350000 USD, revenue 400000 USD, efficiency 0.4 nm per kWh, load 70%. Which profile is it and what should we do?")

You: A new voyage: speed 15 knots, 1200 nm, cost 350000 USD, revenue 400000 USD, efficiency 0.4 nm per kWh, load 70%. Which profile is it and what should we do?
  Step 1: classify_voyage({'speed_knots': 15, 'distance_nm': 1200, 'cost_usd': 350000, 'revenue_usd': 400000, 'efficiency_nm_per_kwh': 0.4, 'load_pct': 70})


**Fleet Copilot:**

This new voyage falls into the **Full but costly** profile, generating an estimated profit of $50,000 on $350,000 in costs and $400,000 in revenue.

**Recommended Action:**
* Investigate high costs: examine fuel pricing, port fees, potential delays, and charter rates to see where expenses can be trimmed.

In [19]:
ask("What is the CII rating of our fleet?")

You: What is the CII rating of our fleet?


**Fleet Copilot:**

I am sorry, but the voyage dataset does not contain Carbon Intensity Indicator (CII) ratings or emissions data, so I cannot provide a CII rating for the fleet.